# Gene Expression Summary: Average Expression & Fraction Expressing

**Goal:** Load the REF1 reference atlas and compute a summary table with:
- `avg_expr` — mean normalized expression per gene, grouped by cell type × time point
- `frac_expr` — fraction of cells with detectable expression (> 0) per gene, grouped by cell type × time point

Paths and loading commands mirror `20260427/power_test_notebook.ipynb`.

## Section 1: Setup

In [1]:
library(monocle3)
library(dplyr)
library(tidyr)
library(tibble)
library(Matrix)

# HDF5 must be loaded before BPCells
dyn.load("/net/gs/vol3/software/modules-sw/hdf5/1.14.3/Linux/Ubuntu22.04/x86_64/lib/libhdf5.so.310")
library(BPCells)

Loading required package: Biobase

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, intersect, is.unsorted, lapply, Map, mapply,
    match, mget, order, paste, pmax, pmax.int, pmin, pmin.int,
    Position, rank, rbind, Reduce, rownames, sapply, setdiff, table,
    tapply, union, unique, unsplit, which.max, which.min


Welcome to Bioconductor

    Vignettes contain introductory material; view with
    'browseVignettes()'. To cite Bioconductor, see
    'citation("Biobase")', and for packages 'citation("pkgname")'.


Loading required package: SingleCellExperiment

Loading required package: SummarizedExperiment

Loading required package: MatrixGenerics

Loading requi

## Section 2: Load Reference Atlas

In [4]:
# --- Paths (from power_test_notebook) ---
temp_path    <- "/net/trapnell/vol1/home/nlammers/tmp_files/nobackup/"
seahub_root  <- "/net/seahub_zfish/vol1/data/annotated/v2.2.0/"
ct_path      <- "/net/trapnell/vol1/home/elizab9/projects/projects/CHEMFISH/resources/unique_ct_full.csv"
dataset_name <- "REF1"
out_dir      <- "/net/trapnell/vol1/home/nlammers/projects/data/morphseq/results/nlammers/20260612/"

# make output directory if it doesn't exist
if (!dir.exists(out_dir)) {
  dir.create(out_dir, recursive = TRUE)
}
# --- Load CDS ---
cds_path <- file.path(seahub_root, dataset_name,
                      paste0(dataset_name, "_projected_cds_v2.2.0/"))

cat("Loading CDS from:", cds_path, "\n")
cds <- load_monocle_objects(
  cds_path,
  matrix_control = list(matrix_class = "BPCells", matrix_path = temp_path)
)
cat("Done.\n")

Loading CDS from: /net/seahub_zfish/vol1/data/annotated/v2.2.0//REF1/REF1_projected_cds_v2.2.0/ 


In [3]:
# Attach broad cell type labels if not already present (same logic as power_test)
ct_broad <- read.csv(ct_path)
ct_broad_filt <- ct_broad %>%
  count(cell_type, cell_type_broad) %>%
  group_by(cell_type) %>%
  slice_max(n, with_ties = FALSE) %>%
  ungroup() %>%
  select(cell_type, cell_type_broad)

md <- as.data.frame(colData(cds))
if (!("cell_type_broad" %in% colnames(md)) && "cell_type" %in% colnames(md)) {
  md <- md %>% left_join(ct_broad_filt, by = "cell_type")
  colData(cds)$cell_type_broad <- md$cell_type_broad
}

cat("colData columns available:\n")
print(colnames(colData(cds)))

colData columns available:
 [1] "cell"                     "Size_Factor"             
 [3] "n.umi"                    "perc_mitochondrial_umis" 
 [5] "intron_fraction"          "P5_barcode"              
 [7] "P7_barcode"               "RT_barcode"              
 [9] "Ligation_barcode"         "RT_plate"                
[11] "hash_umis"                "pval"                    
[13] "qval"                     "top_to_second_best_ratio"
[15] "top_oligo"                "embryo_ID"               
[17] "perturbation"             "target"                  
[19] "type"                     "allele"                  
[21] "strain"                   "expt"                    
[23] "cross_batch"              "collection_batch"        
[25] "dis_protocol"             "fix_protocol"            
[27] "timepoint"                "drug_addition"           
[29] "stage"                    "dose"                    
[31] "dose_type"                "cold_delay"              
[33] "imaging"               

In [ ]:
# Load gene symbol annotations: ENSDARG ID -> gene symbol (common zebrafish name)
# This file has exactly one row per gene in the REF1 atlas (32,031 rows).
gene_annot_path <- "/net/trapnell/vol1/home/nlammers/sci_3lvl_runs/230828_lmx1b_crispant/lmx1b/gene_annotations.txt"

gene_annot <- read.table(
  gene_annot_path, sep = "\t", header = FALSE,
  col.names = c("gene_id", "gene_short_name"),
  stringsAsFactors = FALSE
)

cat(sprintf("Gene annotation rows: %d\n", nrow(gene_annot)))
cat(sprintf("CDS genes covered  : %d / %d\n",
            sum(rownames(cds) %in% gene_annot$gene_id), nrow(cds)))
print(head(gene_annot, 6))

## Section 3: Inspect the Atlas — Feasibility Check

Before committing to a full aggregation across all genes, we check:
- How many genes are in the matrix?
- How many unique (cell_type, timepoint) groups exist?
- How large would the output table be?

The aggregation is a loop over groups, computing `rowMeans` and a fraction-expressing statistic for each. With BPCells backing, the matrix is disk-resident, so each group read is a disk seek — this is the main computational cost.

In [4]:
md_full <- as.data.frame(colData(cds))

n_cells <- ncol(cds)
n_genes <- nrow(cds)

cat(sprintf("Cells  : %s\n", format(n_cells, big.mark = ",")))
cat(sprintf("Genes  : %s\n", format(n_genes, big.mark = ",")))

if ("cell_type" %in% colnames(md_full)) {
  n_ct <- length(unique(na.omit(md_full$cell_type)))
  cat(sprintf("Cell types (fine): %d\n", n_ct))
}
if ("cell_type_broad" %in% colnames(md_full)) {
  n_ct_broad <- length(unique(na.omit(md_full$cell_type_broad)))
  cat(sprintf("Cell types (broad): %d\n", n_ct_broad))
}
if ("timepoint" %in% colnames(md_full)) {
  n_tp <- length(unique(na.omit(md_full$timepoint)))
  cat(sprintf("Timepoints: %d\n", n_tp))
  cat("Values:", paste(sort(unique(na.omit(md_full$timepoint))), collapse = ", "), "\n")
}

Cells  : 739,688
Genes  : 32,031
Cell types (fine): 349
Cell types (broad): 181
Timepoints: 12
Values: 6, 10, 11, 12, 14, 18, 24, 36, 48, 60, 72, 96 


In [5]:
# Parameters that will be used downstream
keep_timepoints <- c(12, 24, 48, 72)
cell_type_col   <- "cell_type_broad"

# Count groups under the chosen filters
group_counts <- md_full %>%
  filter(
    !is.na(.data[[cell_type_col]]),
    !is.na(timepoint),
    timepoint %in% keep_timepoints
  ) %>%
  count(.data[[cell_type_col]], timepoint, name = "n_cells") %>%
  arrange(desc(n_cells))

n_groups <- nrow(group_counts)
cat(sprintf("Cell type column  : %s\n", cell_type_col))
cat(sprintf("Timepoints kept   : %s\n", paste(keep_timepoints, collapse = ", ")))
cat(sprintf("Non-empty groups  : %d\n", n_groups))

n_rows_long <- n_groups * n_genes
est_mem_gb  <- n_rows_long * 3 * 8 / 1e9
cat(sprintf("Estimated rows (long format): %s\n", format(n_rows_long, big.mark = ",")))
cat(sprintf("Estimated memory            : %.2f GB\n", est_mem_gb))

cat("\nGroup size summary (cells per group):\n")
print(summary(group_counts$n_cells))

Cell type column  : cell_type_broad
Timepoints kept   : 12, 24, 48, 72
Non-empty groups  : 646
Estimated rows (long format): 20,692,026
Estimated memory            : 0.50 GB

Group size summary (cells per group):
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
    1.0    13.0    79.0   502.8   324.2 29449.0 


## Section 4: Compute Summary Table

`cell_type_col` and `keep_timepoints` were set in the feasibility cell above. The estimated output is modest (~0.5 GB or less), so no further filtering is needed.

The function iterates over (cell_type_broad × timepoint) groups and computes per-gene `avg_expr` and `frac_expr` from the BPCells-backed matrix.

In [6]:
compute_expression_summary <- function(
  cds,
  cell_type_col       = "cell_type_broad",
  timepoint_col       = "timepoint",
  keep_timepoints     = NULL,        # NULL = all timepoints
  min_cells_per_group = 10
) {
  md <- as.data.frame(colData(cds))
  md$.cell_id <- rownames(md)

  gene_names <- rownames(cds)
  n_genes    <- length(gene_names)
  expr_mat   <- exprs(cds)  # genes x cells (BPCells lazy matrix)

  group_tbl <- md %>%
    dplyr::filter(
      !is.na(.data[[cell_type_col]]),
      !is.na(.data[[timepoint_col]])
    )

  if (!is.null(keep_timepoints)) {
    group_tbl <- group_tbl %>%
      dplyr::filter(.data[[timepoint_col]] %in% keep_timepoints)
  }

  group_tbl <- group_tbl %>%
    dplyr::group_by(.data[[cell_type_col]], .data[[timepoint_col]]) %>%
    dplyr::summarise(
      n_cells = dplyr::n(),
      cells   = list(.cell_id),
      .groups = "drop"
    ) %>%
    dplyr::filter(n_cells >= min_cells_per_group)

  n_groups <- nrow(group_tbl)
  cat(sprintf("Aggregating %d groups x %d genes (min_cells = %d)...\n",
              n_groups, n_genes, min_cells_per_group))
  start_t <- Sys.time()

  results <- vector("list", n_groups)

  for (i in seq_len(n_groups)) {
    if (i == 1 || i %% 25 == 0) {
      elapsed <- as.numeric(difftime(Sys.time(), start_t, units = "mins"))
      cat(sprintf("  group %d/%d  (%.1f min elapsed)\n", i, n_groups, elapsed))
    }

    cells_i <- group_tbl$cells[[i]]
    sub_mat <- expr_mat[, cells_i, drop = FALSE]
    n_i     <- length(cells_i)

    avg_expr  <- as.numeric(Matrix::rowMeans(sub_mat))
    frac_expr <- as.numeric(Matrix::rowSums(sub_mat > 0)) / n_i

    results[[i]] <- tibble::tibble(
      !!cell_type_col := group_tbl[[cell_type_col]][i],
      !!timepoint_col  := group_tbl[[timepoint_col]][i],
      n_cells   = n_i,
      gene      = gene_names,
      avg_expr  = avg_expr,
      frac_expr = frac_expr
    )
  }

  total_min <- as.numeric(difftime(Sys.time(), start_t, units = "mins"))
  cat(sprintf("Done. Total: %.1f min\n", total_min))

  dplyr::bind_rows(results)
}

In [7]:
# Run the aggregation — uses keep_timepoints and cell_type_col set in the feasibility cell
expr_summary <- compute_expression_summary(
  cds,
  cell_type_col       = cell_type_col,
  timepoint_col       = "timepoint",
  keep_timepoints     = keep_timepoints,
  min_cells_per_group = 10
)

cat("\nOutput dimensions:", nrow(expr_summary), "rows x", ncol(expr_summary), "cols\n")
print(head(expr_summary, 10))

Aggregating 501 groups x 32031 genes (min_cells = 10)...
  group 1/501  (0.0 min elapsed)
  group 25/501  (0.2 min elapsed)
  group 50/501  (0.4 min elapsed)
  group 75/501  (0.7 min elapsed)
  group 100/501  (0.8 min elapsed)
  group 125/501  (1.0 min elapsed)
  group 150/501  (1.2 min elapsed)
  group 175/501  (1.2 min elapsed)
  group 200/501  (1.3 min elapsed)
  group 225/501  (1.3 min elapsed)
  group 250/501  (1.4 min elapsed)
  group 275/501  (1.4 min elapsed)
  group 300/501  (1.4 min elapsed)
  group 325/501  (1.5 min elapsed)
  group 350/501  (1.5 min elapsed)
  group 375/501  (1.6 min elapsed)
  group 400/501  (1.7 min elapsed)
  group 425/501  (1.7 min elapsed)
  group 450/501  (1.7 min elapsed)
  group 475/501  (1.7 min elapsed)
  group 500/501  (1.8 min elapsed)
Done. Total: 1.8 min

Output dimensions: 16047531 rows x 6 cols
# A tibble: 10 × 6
   cell_type_broad timepoint n_cells gene               avg_expr frac_expr
   <chr>               <int>   <int> <chr>             

In [ ]:
# Join gene symbols onto the summary table
expr_summary <- expr_summary %>%
  dplyr::left_join(gene_annot, by = c("gene" = "gene_id")) %>%
  dplyr::rename(gene_id = gene) %>%
  dplyr::relocate(gene_short_name, .after = gene_id)

cat("Columns now:", paste(colnames(expr_summary), collapse = ", "), "\n")
print(head(expr_summary, 6))

## Section 5: Save Results

In [8]:
# Save as .rds (fast read-back in R)
# out_rds <- file.path(out_dir, "gene_expression_summary.rds")
# saveRDS(expr_summary, out_rds)
# cat("Saved RDS to:", out_rds, "\n")

# Save as .csv (portable; readable outside R)
out_csv <- file.path(out_dir, "gene_expression_summary.csv")
write.csv(expr_summary, out_csv, row.names = FALSE)
cat("Saved CSV to:", out_csv, "\n")

Saved CSV to: /net/trapnell/vol1/home/nlammers/projects/data/morphseq/results/nlammers/20260612//gene_expression_summary.csv 


In [ ]:
# Quick sanity check: top expressed genes in one cell type x timepoint
sample_group <- expr_summary %>%
  dplyr::filter(!is.na(avg_expr)) %>%
  dplyr::slice(1) %>%
  dplyr::select(all_of(c(cell_type_col, timepoint_col)))

cat("Top 20 genes by avg_expr in:",
    sample_group[[cell_type_col]], "/ timepoint", sample_group[[timepoint_col]], "\n")

expr_summary %>%
  dplyr::filter(
    .data[[cell_type_col]] == sample_group[[cell_type_col]],
    .data[[timepoint_col]] == sample_group[[timepoint_col]]
  ) %>%
  dplyr::arrange(desc(avg_expr)) %>%
  dplyr::select(gene, avg_expr, frac_expr, n_cells) %>%
  head(20) %>%
  print()

## Section 6: Export TF Annotations from zscapetools

`zscapetools::tfs()` returns the lab's curated zebrafish transcription factor list, including Ensembl ID, gene symbol, ZFIN ID, DNA-binding domain family, CIS-BP evidence, and human ortholog annotations.

Run the install cell once to pull the latest version from the cluster before exporting.

In [ ]:
# No install needed — load the TF table directly from the data file
e <- new.env()
load("/net/trapnell/vol1/home/mwu986/seascape-stack/repos/zscapetools/data/tfs.rda", envir = e)
tf_table <- get(ls(e)[1], envir = e)

cat("Object loaded:", ls(e)[1], "\n")
cat("Rows:", nrow(tf_table), "\n")
cat("Cols:", paste(colnames(tf_table), collapse = ", "), "\n")
print(head(tf_table, 6))

In [5]:
# --- Paths (from power_test_notebook) ---
temp_path    <- "/net/trapnell/vol1/home/nlammers/tmp_files/nobackup/"
seahub_root  <- "/net/seahub_zfish/vol1/data/annotated/v2.2.0/"
ct_path      <- "/net/trapnell/vol1/home/elizab9/projects/projects/CHEMFISH/resources/unique_ct_full.csv"
dataset_name <- "REF1"
out_dir      <- "/net/trapnell/vol1/home/nlammers/projects/data/morphseq/results/nlammers/20260612/"

# Save TF table to CSV
out_tf_csv <- file.path(out_dir, "zscapetools_tfs.csv")
write.csv(tf_table, out_tf_csv, row.names = FALSE)
cat("Saved to:", out_tf_csv, "\n")

Saved to: /net/trapnell/vol1/home/nlammers/projects/data/morphseq/results/nlammers/20260612//zscapetools_tfs.csv 
